# Local Differential Privacy (LDP) for Vector Search

A query embedding sent to a vector search engine can be inverted to recover the original
intent, so the client adds calibrated Laplace noise to the vector first. The engine still
returns useful results while no single query reveals what the user was looking for.

## Running locally

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
jupyter lab   # then open this notebook
```

The first run downloads `all-MiniLM-L6-v2` (~90 MB) and caches it under
`~/.cache/huggingface`, and everything after that runs offline.


## 1. Load the embedding model

In [1]:
from sentence_transformers import SentenceTransformer

# Load a pre-trained Sentence Transformer model once for global use.
# Downloaded and cached on first run (~90 MB).
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

/Users/jeff/work/jzonthemtn/ldp-for-search/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 2. Set up the simulated search index

These 20 product names stand in for documents already indexed in OpenSearch. We embed them,
L2-normalize them, then reduce them to 20 dimensions with PCA, and the same PCA model is later
applied to the query so that both live in the same space.


In [2]:
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Fixed seed so the noisy results in this notebook are reproducible.
np.random.seed(42)

# Setup Simulation Data (The "Search Index")
# Imagine these are document embeddings already in OpenSearch
documents = [
    "Laptop",
    "Smartphone",
    "Tablet",
    "Headphones",
    "Monitor",
    "Smartwatch",
    "Wireless Earbuds",
    "Gaming Console",
    "E-Reader",
    "Projector",
    "External Hard Drive",
    "Webcam",
    "Keyboard",
    "Mouse",
    "Printer",
    "Router",
    "Speaker",
    "Microphone",
    "Drone",
    "VR Headset",
]
doc_names = documents

# Generate high-dimensional embeddings for documents
doc_vectors_full = embedding_model.encode(doc_names)
doc_vectors_full = normalize(doc_vectors_full)

# Reduce dimensionality so the noise budget is spread over fewer coordinates
pca_model = PCA(n_components=20)
doc_vectors = pca_model.fit_transform(doc_vectors_full)

print(f"Original document vector dimension: {doc_vectors_full.shape[1]}")
print(f"PCA-reduced document vector dimension: {doc_vectors.shape[1]}")

Original document vector dimension: 384
PCA-reduced document vector dimension: 20


## 3. The LDP engine

This is the logic that would live client-side (e.g. in `ubi.js`). Lower `epsilon` means a
larger noise scale, which means more privacy and less search accuracy.

In [4]:
def inject_laplace_noise(vector, epsilon, sensitivity=1.0):
    """
    Adds Laplace noise to each coordinate of a vector.
    Lower epsilon = Higher Privacy = More Noise.
    """
    scale = sensitivity / epsilon
    noise = np.random.laplace(0, scale, size=vector.shape)
    return vector + noise

## 4. Generate query embeddings

Simulating what a real-world embedding model would produce for a user's typed query.

In [5]:
# Example text queries
text_queries = [
    "laptop computer",
]

# Try these instead to see how different intents behave:
# text_queries = [
#     "laptop for coding",
#     "best smartphone for photography",
#     "comfortable headphones",
#     "gaming monitor review",
# ]

query_embeddings = embedding_model.encode(text_queries)
query_embeddings = normalize(query_embeddings)

print("Generated Embeddings for Queries:")
for i, query in enumerate(text_queries):
    print(f"Query: '{query}'\nEmbedding (first 5 dims): {query_embeddings[i][:5]}...")
    print(f"Embedding Dimension: {len(query_embeddings[i])}")

Generated Embeddings for Queries:
Query: 'laptop computer'
Embedding (first 5 dims): [-0.06790411  0.04620177  0.03757743 -0.04531892  0.0187656 ]...
Embedding Dimension: 384


## 5. Privatize the query vector

Project the raw query through the *fitted* PCA model and then add Laplace noise, so that only
the noised vector ever leaves the client.


In [6]:
# Pick the embedding for 'laptop computer'
raw_query_vector_from_model_full = query_embeddings[0]

epsilon = 1.2  # the privacy control

# Apply the *fitted* PCA model to the raw query vector
raw_query_vector_from_model = pca_model.transform(
    raw_query_vector_from_model_full.reshape(1, -1)
)[0]

# Generate the 'Privacy-Preserving' query on the PCA-reduced vector
noised_query_vector_from_model = inject_laplace_noise(raw_query_vector_from_model, epsilon)

print(f"Original Query ('laptop computer' embedding - full dims): {raw_query_vector_from_model_full[:5]}...")
print(f"PCA-reduced Query (first 5 dims): {raw_query_vector_from_model[:5]}...")
print(f"Noised Query (first 5 dims):      {noised_query_vector_from_model[:5]}...")
print(f"Dimension of noised query vector: {len(noised_query_vector_from_model)}")

Original Query ('laptop computer' embedding - full dims): [-0.06790411  0.04620177  0.03757743 -0.04531892  0.0187656 ]...
PCA-reduced Query (first 5 dims): [ 0.23512647 -0.10999608  0.05589662 -0.1570354   0.01126605]...
Noised Query (first 5 dims):      [-0.00563118  1.82081579  0.57556205  0.02612742 -0.95926113]...
Dimension of noised query vector: 20


/Users/jeff/work/jzonthemtn/ldp-for-search/venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: divide by zero encountered in matmul
  X_transformed = X @ self.components_.T
/Users/jeff/work/jzonthemtn/ldp-for-search/venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: overflow encountered in matmul
  X_transformed = X @ self.components_.T
/Users/jeff/work/jzonthemtn/ldp-for-search/venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:148: RuntimeWarning: invalid value encountered in matmul
  X_transformed = X @ self.components_.T
/Users/jeff/work/jzonthemtn/ldp-for-search/venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:155: RuntimeWarning: divide by zero encountered in matmul
  X_transformed -= xp.reshape(self.mean_, (1, -1)) @ self.components_.T
/Users/jeff/work/jzonthemtn/ldp-for-search/venv/lib/python3.9/site-packages/sklearn/decomposition/_base.py:155: RuntimeWarning: overflow encountered in 

## 6. k-NN search

This simulates the OpenSearch k-NN plugin. Both the documents and the query live in the same
PCA-reduced space, so the noised query can be searched directly.


In [7]:
k_nearest_neighbors = NearestNeighbors(n_neighbors=5, metric='euclidean')
k_nearest_neighbors.fit(doc_vectors)
distances_knn, indices_knn = k_nearest_neighbors.kneighbors([noised_query_vector_from_model])

print("--- Search Results (using noised query vector) ---")
for i, idx in enumerate(indices_knn[0]):
    print(f"Result {i+1}: {doc_names[idx]} (Distance: {distances_knn[0][i]:.4f})")

--- Search Results (using noised query vector) ---
Result 1: Gaming Console (Distance: 4.8870)
Result 2: Webcam (Distance: 4.9303)
Result 3: Mouse (Distance: 4.9658)
Result 4: Smartwatch (Distance: 5.0329)
Result 5: Drone (Distance: 5.0358)


A traditional engine optimizes Precision@1 whereas a privacy-preserving one settles for
Recall@5, accepting that the true intent may sit at position 2 or 3 and still serve the user
without any single query pinning down what they wanted.


## 7. Sweeping epsilon across the privacy and utility tradeoff

One epsilon is one point on a curve, so here we sweep it and measure where the true intent
(`"Laptop"`) ranks, averaged over many noisy draws of the same query.

P@1 counts how often the true intent is the top result, and R@5 counts how often it appears
anywhere in the top 5.

Low epsilon destroys both and high epsilon recovers both while handing the attacker their
answer, so the interesting region is the one in between where R@5 stays high and P@1 has
collapsed.


In [9]:
TRUE_INTENT = "Laptop"
true_idx = doc_names.index(TRUE_INTENT)

epsilons = [0.5, 1.2, 3, 5, 10, 20, 50, 100]
n_trials = 200

# Rank against the whole index so we can find the true intent wherever it lands
sweep_knn = NearestNeighbors(n_neighbors=len(doc_names), metric='euclidean')
sweep_knn.fit(doc_vectors)

np.random.seed(0)  # reproducible sweep

sweep = []
for eps in epsilons:
    ranks = []
    for _ in range(n_trials):
        noised = inject_laplace_noise(raw_query_vector_from_model, eps)
        _, idx = sweep_knn.kneighbors([noised])
        ranks.append(int(np.where(idx[0] == true_idx)[0][0]) + 1)
    ranks = np.array(ranks)
    sweep.append({
        'epsilon': eps,
        'mean_rank': ranks.mean(),
        'p_at_1': (ranks == 1).mean(),
        'r_at_5': (ranks <= 5).mean(),
    })

print(f"Rank of '{TRUE_INTENT}' over {n_trials} noisy queries per epsilon "
      f"(index size: {len(doc_names)})")
print(f"{'epsilon':>8} {'mean rank':>10} {'P@1':>7} {'R@5':>7}")
for row in sweep:
    print(f"{row['epsilon']:>8} {row['mean_rank']:>10.2f} "
          f"{row['p_at_1']:>7.2f} {row['r_at_5']:>7.2f}")

print(f"\nRandom-guess baseline: mean rank {(len(doc_names) + 1) / 2:.1f}, "
      f"P@1 {1 / len(doc_names):.2f}, R@5 {5 / len(doc_names):.2f}")

Rank of 'Laptop' over 200 noisy queries per epsilon (index size: 20)
 epsilon  mean rank     P@1     R@5
     0.5       9.15    0.04    0.27
     1.2       7.05    0.07    0.45
       3       3.94    0.38    0.75
       5       1.84    0.72    0.95
      10       1.03    0.98    1.00
      20       1.00    1.00    1.00
      50       1.00    1.00    1.00
     100       1.00    1.00    1.00

Random-guess baseline: mean rank 10.5, P@1 0.05, R@5 0.25


In [ ]:
# Plot the tradeoff curve
eps_vals = [r['epsilon'] for r in sweep]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(eps_vals, [r['p_at_1'] for r in sweep], 'o-',
        label='P@1 (the correct item is the top hit)')
ax.plot(eps_vals, [r['r_at_5'] for r in sweep], 's-',
        label='R@5 (the correct item is in the top five)')
ax.axhline(5 / len(doc_names), color='gray', linestyle=':',
           label='R@5 if you guessed at random')
ax.axvline(epsilon, color='red', linestyle='dashed', linewidth=2,
           label=f'epsilon {epsilon}, used in the demo above')
ax.set_xscale('log')
# A log axis labels itself 10^0, 10^1, 10^2, which leaves the demo's epsilon 1.2
# — the red line, and the whole point of this chart — sitting between unlabelled
# ticks. Label the sampled values instead.
ax.set_xticks(eps_vals)
ax.set_xticklabels([f'{t:g}' for t in eps_vals])
ax.minorticks_off()
ax.set_xlabel('Epsilon (higher = less noise, less privacy)')
ax.set_ylabel(f"Fraction of noisy queries that find '{TRUE_INTENT}'")
ax.set_ylim(-0.05, 1.05)
ax.set_title(f"Does the noised query still find '{TRUE_INTENT}'?")
ax.legend()
ax.grid(alpha=0.3)
fig.savefig('plots/07_epsilon_tradeoff_toy_index.png', dpi=200, bbox_inches='tight')
plt.show()


## 8. The statistical tent audit

Draw 1,000 independent noisy versions of the same query and histogram the first coordinate,
which should produce the characteristic Laplace tent centered on the true value. This is the
auditable proof that the noise mechanism behaves as claimed.


In [ ]:
def plot_audit():
    # Generate 1000 noisy samples to verify the Laplace distribution
    samples = [
        inject_laplace_noise(raw_query_vector_from_model, epsilon)[0]
        for _ in range(1000)
    ]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(samples, bins=50, color='skyblue', edgecolor='black', alpha=0.7)
    ax.axvline(
        raw_query_vector_from_model[0],
        color='red', linestyle='dashed', linewidth=2,
        label='the true value, before noise',
    )
    ax.set_title(
        f"One coordinate of one query, privatized 1,000 times (epsilon = {epsilon})"
    )
    ax.set_xlabel('Value of that coordinate after noise')
    ax.set_ylabel('Number of draws, out of 1,000')
    ax.legend()
    fig.savefig('plots/08_laplace_tent_audit.png', dpi=200, bbox_inches='tight')
    plt.show()


plot_audit()


In [ ]:
# The same audit at two epsilons. The x axis is shared on purpose: let each panel
# autoscale and both look like the same tent, which is exactly the wrong lesson.
COMPARE_EPSILON = 10

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)
draws = {
    eps: [inject_laplace_noise(raw_query_vector_from_model, eps)[0] for _ in range(1000)]
    for eps in (epsilon, COMPARE_EPSILON)
}
lo, hi = min(draws[epsilon]), max(draws[epsilon])
edges = np.linspace(lo, hi, 51)

for ax, eps in zip(axes, (epsilon, COMPARE_EPSILON)):
    ax.hist(draws[eps], bins=edges, color='skyblue', edgecolor='black', alpha=0.7)
    ax.axvline(
        raw_query_vector_from_model[0],
        color='red', linestyle='dashed', linewidth=2,
        label='the true value, before noise',
    )
    ax.set_title(f"epsilon = {eps}   (noise scale {1 / eps:.3f})")
    ax.set_xlabel('Value of that coordinate after noise')
    ax.grid(alpha=0.3)

axes[0].set_ylabel('Number of draws, out of 1,000')
axes[0].legend(loc='upper left')
fig.suptitle('Same coordinate, same 1,000 draws, two privacy settings')
fig.tight_layout()
fig.savefig('plots/08b_epsilon_spread_comparison.png', dpi=200, bbox_inches='tight')
plt.show()


## 9. Privacy failure vs. privacy success

The same inversion attack runs against both vectors, raw and noised. The attack is simulated
by k-NN search against the document index, so that given a vector it finds the nearest
documents and reads off the intent.


In [8]:
# Simulate 'Privacy Failure': Vector Inversion on Raw Query
k_nearest_neighbors_raw = NearestNeighbors(n_neighbors=2, metric='euclidean')
k_nearest_neighbors_raw.fit(doc_vectors)
distances_raw, indices_raw = k_nearest_neighbors_raw.kneighbors([raw_query_vector_from_model])

print("--- Privacy Failure: Inferred from RAW Query Vector ---")
print("Attacker's 'guess' of user's intent (high accuracy):")
for i, idx in enumerate(indices_raw[0]):
    print(f"Result {i+1}: {doc_names[idx]} (Distance: {distances_raw[0][i]:.4f})")

# Simulate 'Privacy Success': Vector Inversion on LDP-Noised Query
k_nearest_neighbors_noised = NearestNeighbors(n_neighbors=2, metric='euclidean')
k_nearest_neighbors_noised.fit(doc_vectors)
distances_noised, indices_noised = k_nearest_neighbors_noised.kneighbors(
    [noised_query_vector_from_model]
)

print("\n--- Privacy Success: Inferred from LDP-Noised Query Vector ---")
print("Attacker's 'guess' of user's intent (low accuracy/gibberish):")
for i, idx in enumerate(indices_noised[0]):
    print(f"Result {i+1}: {doc_names[idx]} (Distance: {distances_noised[0][i]:.4f})")

--- Privacy Failure: Inferred from RAW Query Vector ---
Attacker's 'guess' of user's intent (high accuracy):
Result 1: Laptop (Distance: 0.2092)
Result 2: Keyboard (Distance: 0.9104)

--- Privacy Success: Inferred from LDP-Noised Query Vector ---
Attacker's 'guess' of user's intent (low accuracy/gibberish):
Result 1: Gaming Console (Distance: 4.8870)
Result 2: Webcam (Distance: 4.9303)


---

## 10. Scaling up to a real 43,000-product index

The 20-item index is small enough to hold in your head, but it flatters the privacy result.
With 20 unrelated items, noise pushes the query to a *random* item, so "the attacker learned
nothing" is partly an artifact of having no near-neighbours.

Here we swap in WANDS, Wayfair's product search relevance dataset (ECIR 2022, MIT licensed),
which holds 42,994 real products across 861 classes. The mechanism does not change, only the
index does.

WANDS ships a `product_class` per product, which separates two very different questions.
Item-level recovery asks whether the attacker recovered the *exact product*, and class-level
recovery asks whether they merely recovered the *category* such as Beds or Area Rugs. That
distinction is invisible in a toy index, and it is the one that matters.

> Run `python download_wands.py` once before this section.


### 10a. Fetch and cache the dataset

Running `download_wands.py` downloads the WANDS files, embeds 43k product names, fits the PCA
basis, and caches everything to `data/` as numpy files.

It uses this kernel's interpreter and needs nothing beyond what sections 1 to 9 already use,
and it is idempotent, so anything cached is skipped. Expect 30 to 60 seconds on the first run.

Run it before presenting rather than during, because after that the notebook needs no network.


In [13]:
import subprocess
import sys

# The script runs in the SAME interpreter as this kernel, so whatever works here works there
print(f"Kernel interpreter: {sys.executable}\n")

process = subprocess.Popen(
    [sys.executable, 'download_wands.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line.rstrip())

if process.wait() != 0:
    raise RuntimeError(
        "download_wands.py failed. Check the output above. The usual causes are no network "
        "access, or a kernel whose interpreter is missing numpy, scikit-learn or "
        "sentence-transformers. The path printed above is the interpreter it used."
    )

Kernel interpreter: /work/jzonthemtn/ldp-for-search/venv/bin/python3

product.csv: already cached
query.csv: already cached
embeddings: already cached

Done. The notebook can now run fully offline.


In [14]:
from pathlib import Path

DATA_DIR = Path('data')

if not (DATA_DIR / 'wands_vectors_pca20.npy').exists():
    raise FileNotFoundError("WANDS cache not found. Run the cell above first.")

# Pre-computed by download_wands.py: same model, same PCA dimensionality as above.
# Stored as plain .npy/.npz so section 10 needs no libraries beyond the ones already in use.
wands_vectors = np.load(DATA_DIR / 'wands_vectors_pca20.npy')
wands_pca = np.load(DATA_DIR / 'wands_pca.npz')

_products = np.load(DATA_DIR / 'wands_products.npz')
wands_names = _products['names']
wands_classes = _products['classes']

print(f"Index size:      {len(wands_vectors):,} products")
print(f"Product classes: {len(np.unique(wands_classes)):,}")
print(f"Vector shape:    {wands_vectors.shape}")
print("\nSample products:")
for name in wands_names[:5]:
    print(f"  - {name}")

Index size:      42,994 products
Product classes: 861
Vector shape:    (42994, 20)

Sample products:
  - solid wood platform bed
  - all-clad 7 qt . slow cooker
  - all-clad electrics 6.5 qt . slow cooker
  - all-clad all professional tools pizza cutter
  - baldwin prestige alcott passage knob with round rosette


### 10b. The attack at scale

The baseline first, which is what an attacker recovers from the raw query vector against a
real index. This is the same vector inversion simulation as section 9, just with 43k
candidates instead of 20.


In [15]:
def wands_project(text):
    """Embed a query and project it into the cached WANDS PCA space."""
    full = normalize(embedding_model.encode([text]))[0]
    return (full - wands_pca['mean']) @ wands_pca['components'].T


def wands_topk(vector, k=5):
    """Return the indices of the k nearest products, nearest first."""
    distances = np.linalg.norm(wands_vectors - vector, axis=1)
    top = np.argpartition(distances, k)[:k]
    return top[np.argsort(distances[top])], distances


WANDS_QUERY = "solid wood platform bed"

wands_raw_query = wands_project(WANDS_QUERY)
raw_top, raw_distances = wands_topk(wands_raw_query, k=5)

# The attacker's target: what the RAW query resolves to
true_item_name = wands_names[raw_top[0]]
true_item_class = wands_classes[raw_top[0]]

print(f"Query: '{WANDS_QUERY}'")
print("\n--- Privacy Failure: attacker inverts the RAW query vector ---")
for rank, idx in enumerate(raw_top, start=1):
    print(f"  {rank}. {wands_names[idx][:52]:<52} "
          f"[{wands_classes[idx][:22]}]  d={raw_distances[idx]:.3f}")

print(f"\nTrue intent -> item:  '{true_item_name}'")
print(f"            -> class: '{true_item_class}'")

Query: 'solid wood platform bed'

--- Privacy Failure: attacker inverts the RAW query vector ---
  1. solid wood platform bed                              [Beds]  d=0.000
  2. solid wood platform bed                              [Beds]  d=0.000
  3. devery solid wood platform bed                       [Beds]  d=0.099
  4. wynd solid wood platform bed                         [Beds]  d=0.146
  5. travie solid wood platform bed                       [Beds]  d=0.149

True intent -> item:  'solid wood platform bed'
            -> class: 'Beds'


### 10c. Item-level vs class-level recovery

Sweep epsilon again, scoring the attacker on both questions. A match counts at the item level
when the noised query's top hit has the same product name as the raw query's top hit (WANDS
has genuine duplicate listings, so comparing names avoids penalising the attacker for landing
on an identical product), and at the class level when it merely lands in the same category.

Note the `scale` column. Laplace scale is `sensitivity / epsilon`, and what matters is its
size *relative to the spread of the vector coordinates*. That spread differs between the two
indexes, which is why the useful epsilon here is nothing like `1.2`, and why epsilon is not
portable.


In [16]:
wands_epsilons = [1, 5, 10, 20, 30, 50, 75, 100, 200]
n_trials = 300

coord_std = wands_vectors.std(axis=0).mean()
print(f"Mean per-coordinate spread of the index: {coord_std:.3f}")
print(f"Query: '{WANDS_QUERY}'  |  {n_trials} noisy draws per epsilon\n")

header = f"{'epsilon':>8}{'scale':>8}{'item P@1':>10}{'item R@5':>10}{'class P@1':>11}{'class R@5':>11}"
print(header)
print('-' * len(header))

wands_sweep = []
for eps in wands_epsilons:
    rng = np.random.default_rng(0)  # same draws at every epsilon, so rows are comparable
    noised = wands_raw_query + rng.laplace(0, 1.0 / eps, size=(n_trials, wands_vectors.shape[1]))

    # Distance from every noisy query to every product, then the top 5 of each row
    dists = np.linalg.norm(wands_vectors[None, :, :] - noised[:, None, :], axis=2)
    top5 = np.argpartition(dists, 5, axis=1)[:, :5]
    top5 = np.take_along_axis(top5, np.argsort(np.take_along_axis(dists, top5, 1), axis=1), axis=1)
    top1 = top5[:, 0]

    row = {
        'epsilon': eps,
        'scale': 1.0 / eps,
        'item_p1': (wands_names[top1] == true_item_name).mean(),
        'item_r5': (wands_names[top5] == true_item_name).any(axis=1).mean(),
        'class_p1': (wands_classes[top1] == true_item_class).mean(),
        'class_r5': (wands_classes[top5] == true_item_class).any(axis=1).mean(),
    }
    wands_sweep.append(row)
    print(f"{eps:>8}{row['scale']:>8.3f}{row['item_p1']:>10.2f}{row['item_r5']:>10.2f}"
          f"{row['class_p1']:>11.2f}{row['class_r5']:>11.2f}")

n_docs = len(wands_vectors)
class_size = int((wands_classes == true_item_class).sum())
print(f"\nRandom-guess baseline over {n_docs:,} products:")
print(f"  item P@1  {1 / n_docs:.5f}   class P@1  {class_size / n_docs:.5f} "
      f"('{true_item_class}' has {class_size:,} members)")

Mean per-coordinate spread of the index: 0.128
Query: 'solid wood platform bed'  |  300 noisy draws per epsilon

 epsilon   scale  item P@1  item R@5  class P@1  class R@5
----------------------------------------------------------
       1   1.000      0.00      0.01       0.06       0.11
       5   0.200      0.04      0.09       0.33       0.60
      10   0.100      0.13      0.31       0.63       0.92
      20   0.050      0.42      0.76       0.82       1.00
      30   0.033      0.70      0.96       0.93       1.00
      50   0.020      0.93      1.00       0.98       1.00
      75   0.013      0.99      1.00       1.00       1.00
     100   0.010      1.00      1.00       1.00       1.00
     200   0.005      1.00      1.00       1.00       1.00

Random-guess baseline over 42,994 products:
  item P@1  0.00002   class P@1  0.02586 ('Beds' has 1,112 members)


In [ ]:
# The gap between the two curves IS the privacy story
eps_vals = [r['epsilon'] for r in wands_sweep]

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(eps_vals, [r['class_p1'] for r in wands_sweep], 's-', color='tab:orange',
        label='the category the user searched in (Beds, Area Rugs, ...)')
ax.plot(eps_vals, [r['item_p1'] for r in wands_sweep], 'o-', color='tab:blue',
        label='the exact product the user searched for')

ax.fill_between(
    eps_vals,
    [r['item_p1'] for r in wands_sweep],
    [r['class_p1'] for r in wands_sweep],
    color='tab:orange', alpha=0.15,
    label='the gap, where the category is known but the item is not',
)

ax.set_xscale('log')
# A log axis labels itself 10^0, 10^1, 10^2, and the slide captions cite specific
# epsilons. Label the sampled values instead, so "at epsilon 1" is readable off
# the axis rather than converted in the audience's head.
eps_ticks = [1, 5, 10, 20, 50, 100, 200]
ax.set_xticks(eps_ticks)
ax.set_xticklabels([str(t) for t in eps_ticks])
ax.minorticks_off()
ax.set_xlabel('Epsilon (higher = less noise, less privacy)')
ax.set_ylabel('Fraction of queries the attacker recovers')
ax.set_ylim(-0.05, 1.05)
ax.set_title(f"What an attacker recovers from a noised query ({len(wands_vectors):,} products)")
ax.legend(loc='upper left')
ax.grid(alpha=0.3)
fig.savefig('plots/10_item_vs_class_recovery.png', dpi=200, bbox_inches='tight')
plt.show()


### What the larger index shows that the toy index could not

The two curves separate, and around epsilon 10 to 20 the attacker recovers the *category* most
of the time while the *exact product* stays hidden. They can tell this person was shopping for
beds, but not which bed.

That gap is the honest version of the privacy claim, and more defensible than "the attacker
gets gibberish". It also states the residual leak plainly, because LDP at a usable epsilon
does not hide the broad category. If category alone is sensitive, a medical corpus rather than
furniture, this mechanism at this epsilon is not sufficient.


---

## 11. What survives, which is aggregate trends

Every result so far measures what LDP *costs*, since section 7 showed P@1 collapsing and
section 10 showed the attacker losing the exact item. Taken alone that is pure loss, and a
strange argument for adopting the technique.

The reason it is useful is the asymmetry. The noise is zero-mean, so it cancels when you
average over many independent users, and one person's query is unrecoverable while the trend
across ten thousand people is not.

Learning-to-rank training data, query-intent clustering and demand trends are all aggregate
statistics, and none of them need an individual query to be readable.

This section makes that concrete. We build cohorts from the 480 real WANDS queries, have every
user privatize on their own device, then ask two questions of the same noised data. Can we
recover what any individual searched for, which should fail, and can we recover what the
cohort was shopping for, which should succeed?


In [18]:
import csv
from collections import Counter, defaultdict

# The 480 real user queries WANDS ships, grouped by their annotated class
with open(DATA_DIR / 'query.csv', newline='', encoding='utf-8') as handle:
    query_rows = list(csv.DictReader(handle, delimiter='\t'))

queries_by_class = defaultdict(list)
for row in query_rows:
    queries_by_class[row['query_class']].append(row['query'])

# Use the five best-populated classes as our user cohorts
COHORTS = [c for c, _ in Counter(
    {k: len(v) for k, v in queries_by_class.items()}).most_common(5)]

# Embed every query once, then project into the same PCA space as the index
cohort_queries = [q for c in COHORTS for q in queries_by_class[c]]
cohort_embedded = ((normalize(embedding_model.encode(cohort_queries)) - wands_pca['mean'])
                   @ wands_pca['components'].T)
query_vectors = dict(zip(cohort_queries, cohort_embedded))

print(f"{len(query_rows)} real queries, {len(queries_by_class)} classes. Using 5 as cohorts:\n")
for c in COHORTS:
    sample = ', '.join(queries_by_class[c][:3])
    print(f"  {c:<28} {len(queries_by_class[c]):>2} distinct queries   e.g. {sample}")

480 real queries, 189 classes. Using 5 as cohorts:

  Wall Art                     20 distinct queries   e.g. sunflower, 70s inspired furniture, wall art fiji
  Accent Chairs                16 distinct queries   e.g. leather chairs, tufted chair with gold legs, sancroft armchair
  Beds                         15 distinct queries   e.g. king poster bed, beds that have leds, full metal bed rose gold
  Area Rugs                    15 distinct queries   e.g. ombre rug, tollette teal outdoor rug, regner power loom red
  Coffee & Cocktail Tables     10 distinct queries   e.g. smart coffee table, westling coffee table, unique coffee tables


### 11a. One user hides, the crowd does not

Each simulated user picks one real query from their cohort and adds Laplace noise on their own
device, so the server only ever sees noised vectors. Both readouts below come from that same
data at `epsilon = 1.0`, which is heavy noise by the standards of section 10.

The cohort is identified by averaging its noised vectors and matching to the nearest cohort
centroid. The dominant-class readout asks the same question more plainly, which is that of the
25 products nearest the recovered centroid, most of them belong to one category.


In [19]:
AGG_EPSILON = 1.0     # heavy noise: section 10 showed this leaks almost nothing per query
USERS_PER_COHORT = 20_000

rng = np.random.default_rng(0)

# Noiseless cohort centroids, used only to score the recovery
true_centroids = {c: np.mean([query_vectors[q] for q in queries_by_class[c]], axis=0)
                  for c in COHORTS}


def dominant_class(vector, k=25):
    """The category most common among the k products nearest this point."""
    distances = np.linalg.norm(wands_vectors - vector, axis=1)
    nearest = np.argpartition(distances, k)[:k]
    return Counter(wands_classes[nearest]).most_common(1)[0][0]


print(f"epsilon = {AGG_EPSILON}, {USERS_PER_COHORT:,} users per cohort\n")
header = f"{'cohort':<28}{'individual':>12}{'cohort ID':>12}   {'dominant class of recovered centroid'}"
print(header)
print('-' * len(header))

identified = 0
for cohort in COHORTS:
    picks = rng.choice(queries_by_class[cohort], size=USERS_PER_COHORT)
    raw = np.array([query_vectors[q] for q in picks])

    # Every user privatizes independently, client-side
    noised = raw + rng.laplace(0, 1.0 / AGG_EPSILON, size=raw.shape)

    # Readout 1: try to recover what a single user searched for
    individual_hits = [
        wands_classes[int(np.argmin(np.linalg.norm(wands_vectors - noised[i], axis=1)))] == cohort
        for i in range(300)
    ]

    # Readout 2: average the cohort, then match to the nearest cohort centroid
    centroid = noised.mean(axis=0)
    matched = min(COHORTS, key=lambda c: np.linalg.norm(centroid - true_centroids[c]))
    identified += matched == cohort

    print(f"{cohort:<28}{np.mean(individual_hits):>12.3f}"
          f"{('OK' if matched == cohort else 'MISS'):>12}   {dominant_class(centroid)}")

print(f"\nCohorts correctly identified from noised data: {identified}/{len(COHORTS)}")
print("Individual recovery stays near the random baseline of "
      f"{1 / len(np.unique(wands_classes)):.4f} throughout.")

epsilon = 1.0, 20,000 users per cohort

cohort                        individual   cohort ID   dominant class of recovered centroid
-------------------------------------------------------------------------------------------
Wall Art                           0.020          OK   Wall Art
Accent Chairs                      0.020          OK   Dining Chairs
Beds                               0.063          OK   Beds
Area Rugs                          0.070          OK   Area Rugs
Coffee & Cocktail Tables           0.030          OK   Coffee & Cocktail Tables

Cohorts correctly identified from noised data: 5/5
Individual recovery stays near the random baseline of 0.0012 throughout.


### 11b. Why it works, because the noise averages away

The estimate improves as `1 / sqrt(n)`, which is not a property of this dataset but the
standard error of a mean, and it is what makes LDP practical at population scale. Doubling
your privacy budget is expensive, whereas collecting four times as many users costs nothing
extra in privacy and halves your error.


In [ ]:
sample_sizes = [10, 30, 100, 300, 1_000, 3_000, 10_000, 30_000, 100_000]
REPEATS = 5
CONV_COHORT = 'Beds'

rng = np.random.default_rng(1)
pool = queries_by_class[CONV_COHORT]

errors = []
for n in sample_sizes:
    trial_errors = []
    for _ in range(REPEATS):
        raw = np.array([query_vectors[q] for q in rng.choice(pool, size=n)])
        noised = raw + rng.laplace(0, 1.0 / AGG_EPSILON, size=raw.shape)
        trial_errors.append(np.linalg.norm(noised.mean(axis=0) - raw.mean(axis=0)))
    errors.append(np.mean(trial_errors))

# Theoretical 1/sqrt(n) curve, anchored at the first measured point
reference = [errors[0] * (sample_sizes[0] / n) ** 0.5 for n in sample_sizes]

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.loglog(sample_sizes, errors, 'o-', color='tab:green',
          label='how far the noised average lands from the truth')
ax.loglog(sample_sizes, reference, '--', color='gray',
          label='what theory predicts, proportional to 1/sqrt(n)')
# The question this chart answers is "how many users do you need", so mark the
# answer on the x axis rather than drawing the accuracy bar on the y axis and
# making the room read the crossing sideways. The bar itself is unchanged: the
# mean per-coordinate spread of the index, a deliberately strict target.
accuracy_target = wands_vectors.std(axis=0).mean()
_x, _y = np.array(sample_sizes, float), np.array(errors)
_i = int(np.argmax(_y < accuracy_target))
smallest_segment = float(np.exp(np.interp(
    np.log(accuracy_target), np.log(_y[[_i, _i - 1]]), np.log(_x[[_i, _i - 1]]))))
ax.axvline(smallest_segment, color='tab:red', linestyle='dashed', linewidth=2,
           label=f'{round(smallest_segment, -2):,.0f} users, the smallest measurable segment')
# A log axis labels itself 10^1 ... 10^5, which leaves the sampled 30, 300, 3k
# and 30k between unlabelled decades. Label the sizes actually measured.
ax.set_xticks(sample_sizes)
ax.set_xticklabels([f'{n}' if n < 1000 else f'{n // 1000:g}k' for n in sample_sizes])
ax.tick_params(axis='x', which='minor', bottom=False, labelbottom=False)
ax.set_xlabel('Users whose noised queries are averaged together')
ax.set_ylabel("Distance from the segment's true center")
# A log y axis labels itself 10^0 and 10^-1 and then fills the gaps with minor
# gridlines at 2, 3, 4 and so on, which bunch up and mark values nobody can read.
# Label the decade fractions instead and keep gridlines only where there is a label.
err_ticks = [0.02, 0.05, 0.1, 0.2, 0.5, 1, 2]
ax.set_yticks(err_ticks)
ax.set_yticklabels([f'{t:g}' for t in err_ticks])
ax.tick_params(axis='y', which='minor', left=False, labelleft=False)
ax.set_title(f"Averaging more users recovers the truth (epsilon = {AGG_EPSILON}, segment '{CONV_COHORT}')")
ax.legend()
ax.grid(alpha=0.3)
fig.savefig('plots/11_aggregate_convergence.png', dpi=200, bbox_inches='tight')
plt.show()

print(f"{'users':>8}{'centroid error':>17}")
for n, e in zip(sample_sizes, errors):
    print(f"{n:>8,}{e:>17.4f}")


### What this means for relevance work

The red line marks where a segment becomes big enough to measure, and past it the recovered
centroid points at a specific neighbourhood of the catalogue rather than a vague direction.

So the constraint is not that you lose your analytics, it is that you lose the ability to ask
about one person. Anything you can phrase as a population statistic still works, and those are
the inputs relevance tuning depends on.

Two honest limits are worth stating alongside that. The dominant-class readout puts the Accent
Chairs cohort in an adjacent chair category, which is the same category blurring section 10
measured, and it does not sharpen with more users because it is bias rather than noise.
Low-traffic segments also never reach the population sizes this depends on, so the tail stays
unmeasurable.


---

## 12. What segmenting costs a ranker

Section 11 showed that a segment centroid is accurate once enough users contribute. That
leaves the question a relevance engineer will actually ask, which is what you give up by
ranking against a segment instead of against the query someone typed.

WANDS ships graded relevance judgments, 233,448 of them across the 480 queries, labelled
Exact, Partial or Irrelevant. So we can rank each query's judged products three ways and
score NDCG@10 against those labels. The three runs differ in one thing only, which is the
vector we rank against.

1. The query's own vector, which is what a conventional ranker uses.
2. Its segment centroid, computed from the raw queries in that segment.
3. Its segment centroid recovered from noised queries at `epsilon = 1.0`.

The gap between 1 and 2 is what segmenting costs. The gap between 2 and 3 is what privacy
costs. Sweeping the number of segments shows how both move as segments get coarser.


In [ ]:
import csv
from collections import defaultdict
from sklearn.cluster import KMeans
from sklearn.metrics import ndcg_score

LTR_EPSILON = 1.0
LTR_USERS = 20_000          # per segment, comfortably past the threshold section 11 measured
SEGMENT_COUNTS = [5, 10, 20, 40, 80, 160, 320]

_ids = np.load(DATA_DIR / 'wands_products.npz')['ids']
row_of = {pid: i for i, pid in enumerate(_ids)}

query_text = {r['query_id']: r['query'] for r in
              csv.DictReader(open(DATA_DIR / 'query.csv', newline='', encoding='utf-8'),
                             delimiter='\t')}

GRADE = {'Exact': 2, 'Partial': 1, 'Irrelevant': 0}
judged = defaultdict(list)
with open(DATA_DIR / 'label.csv', newline='', encoding='utf-8') as handle:
    for r in csv.DictReader(handle, delimiter='\t'):
        row = row_of.get(r['product_id'])
        if row is not None:
            judged[r['query_id']].append((row, GRADE[r['label']]))

# A query needs a few candidates and at least one relevant one for NDCG to mean anything.
ltr_qids = [q for q in query_text
            if len(judged[q]) >= 10 and any(g for _, g in judged[q])]
ltr_vectors = dict(zip(ltr_qids,
                       (normalize(embedding_model.encode([query_text[q] for q in ltr_qids]))
                        - wands_pca['mean']) @ wands_pca['components'].T))


def mean_ndcg(representation):
    """NDCG@10 when each query's candidates are ranked by distance to its representation."""
    scores = []
    for q in ltr_qids:
        rows, grades = zip(*judged[q])
        distance = np.linalg.norm(wands_vectors[list(rows)] - representation[q], axis=1)
        scores.append(ndcg_score([list(grades)], [-distance], k=10))
    return float(np.mean(scores))


rng = np.random.default_rng(0)
per_query = mean_ndcg(ltr_vectors)
X = np.array([ltr_vectors[q] for q in ltr_qids])

print(f"{len(ltr_qids)} queries scored against {sum(len(judged[q]) for q in ltr_qids):,} judgments")
print(f"Ranking on the query itself: NDCG@10 {per_query:.3f}\n")
header = f"{'segments':>9}{'queries each':>14}{'centroid':>10}{'noised':>9}"
print(header); print('-' * len(header))

ltr_curve = []
for k in SEGMENT_COUNTS:
    labels = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)
    members = defaultdict(list)
    for q, c in zip(ltr_qids, labels):
        members[c].append(q)

    clean, noised = {}, {}
    for c, qs in members.items():
        pool = np.array([ltr_vectors[q] for q in qs])
        # Every simulated user privatizes one query from the segment, on their own device
        picks = pool[rng.integers(0, len(pool), size=LTR_USERS)]
        noised_centroid = (picks + rng.laplace(0, 1.0 / LTR_EPSILON,
                                               size=picks.shape)).mean(axis=0)
        for q in qs:
            clean[q], noised[q] = pool.mean(axis=0), noised_centroid

    a, b = mean_ndcg(clean), mean_ndcg(noised)
    ltr_curve.append((k, a, b))
    print(f"{k:>9}{len(ltr_qids) / k:>14.1f}{a:>10.3f}{b:>9.3f}")


In [ ]:
# The finding is that two of these are the same height, so bars say it faster than
# lines that overlap. Forty segments is roughly twelve queries each in this corpus.
SHOW_AT = 40
clean_at, noised_at = next((a, b) for k, a, b in ltr_curve if k == SHOW_AT)

labels = ['Ranking on\nthe query', 'Ranking on\nthe segment', 'Ranking on the\nnoisy segment']
values = [per_query, clean_at, noised_at]
colours = ['tab:blue', 'tab:orange', 'tab:green']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(labels, values, color=colours, width=0.55)
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, v + 0.012, f'{v:.2f}',
            ha='center', fontsize=14, fontweight='bold')

ax.set_ylim(0, 0.95)
ax.set_ylabel(f'Mean NDCG@10 across {len(ltr_qids)} queries')
ax.grid(alpha=0.3, axis='y')
fig.savefig('plots/12_ltr_segmentation.png', dpi=200, bbox_inches='tight')
plt.show()


### Reading the result

The orange and green lines sit on top of each other, which is the finding. At 20,000 users a
segment the noise costs almost nothing, so nearly all the distance from the blue line is the
price of segmenting rather than the price of privacy.

That matters because segmenting is a modelling choice you can evaluate on clean data, with no
privacy machinery involved at all, and it is recoverable by using finer segments wherever you
have the traffic to support them.

Two caveats worth carrying. This ranks by vector distance alone rather than training a model,
so it isolates the query representation instead of measuring a production ranker. And 480
queries is a small corpus, so treat the numbers as indicative.
